# 01 — Pembersihan Data Olist

**Tujuan:** memeriksa kualitas data dan menyiapkan tabel yang digunakan untuk tiga pertanyaan bisnis proyek Olist.

Notebook ini mengikuti tahap Minggu 3 pada pedoman proyek: pemeriksaan nilai kosong, duplikasi, tipe tanggal, serta pembuatan kolom turunan berupa bulan pesanan, lama pengiriman, dan penanda keterlambatan.

**Prinsip pengerjaan**
- File pada `Dataset/Raw` tidak diubah.
- Baris tidak dihapus otomatis hanya karena memiliki nilai kosong atau terdeteksi duplikat.
- Setiap anomali dicatat agar keputusan pembersihan dapat dijelaskan.
- Data hasil pemrosesan disimpan secara lokal di `Dataset/Processed/Cleaned` dan tidak perlu diunggah ke GitHub.


## 1. Persiapan dan lokasi proyek

Jalankan notebook dari folder proyek atau dari subfolder `notebooks`. Kode akan mencari folder proyek yang memiliki `Dataset/Raw`.

Dataset yang digunakan dibatasi pada tabel yang mendukung tiga pertanyaan bisnis:
1. Penjualan dan kategori produk: `orders`, `order_items`, `products`, dan terjemahan kategori.
2. Keterlambatan dan ulasan: `orders` dan `order_reviews`.
3. Pembayaran: `order_payments`.


In [2]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_project_root(start_path: Path) -> Path:
    """Cari root proyek berdasarkan keberadaan Dataset/Raw."""
    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if (candidate / "Dataset" / "Raw").is_dir():
            return candidate

    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. "
        "Pastikan notebook berada di dalam folder Mini Project Olist."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "Dataset" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Dataset" / "Processed"
CLEANED_DIR = PROCESSED_DIR / "Cleaned"

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DIR)
print("Output       :", CLEANED_DIR)
print("Pandas       :", pd.__version__)


Project root : /mnt/c/Users/ASUS/Github/Mini Project Olist
Raw data     : /mnt/c/Users/ASUS/Github/Mini Project Olist/Dataset/Raw
Output       : /mnt/c/Users/ASUS/Github/Mini Project Olist/Dataset/Processed/Cleaned
Pandas       : 3.1.1


## 2. Membaca dataset

File review menggunakan TSV hasil konversi yang sudah divalidasi: **7 kolom dan 99.224 record**. Lima dataset lain dibaca dari CSV asli.

Jika salah satu file belum tersedia atau nama file berbeda, hentikan proses dan periksa isi folder `Dataset/Raw`; jangan mengganti nama kolom atau file berdasarkan asumsi.


In [3]:
FILE_PATHS = {
    "orders": RAW_DIR / "olist_orders_dataset.csv",
    "order_items": RAW_DIR / "olist_order_items_dataset.csv",
    "products": RAW_DIR / "olist_products_dataset.csv",
    "category_translation": RAW_DIR / "product_category_name_translation.csv",
    "payments": RAW_DIR / "olist_order_payments_dataset.csv",
}

REVIEWS_TSV = PROCESSED_DIR / "olist_order_reviews_dataset.tsv"
REVIEWS_CSV = RAW_DIR / "olist_order_reviews_dataset.csv"

if REVIEWS_TSV.exists():
    FILE_PATHS["reviews"] = REVIEWS_TSV
    REVIEWS_SEPARATOR = "\t"
elif REVIEWS_CSV.exists():
    FILE_PATHS["reviews"] = REVIEWS_CSV
    REVIEWS_SEPARATOR = ","
else:
    raise FileNotFoundError(
        "Dataset review tidak ditemukan. Periksa folder Dataset/Raw "
        "dan Dataset/Processed."
    )

missing_files = [
    str(path)
    for name, path in FILE_PATHS.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "File dataset berikut belum ditemukan:\n- "
        + "\n- ".join(missing_files)
    )


def read_dataset(path: Path, separator: str = ",") -> pd.DataFrame:
    """Baca dataset sebagai string agar ID dan nilai asli tetap terjaga."""
    return pd.read_csv(
        path,
        sep=separator,
        dtype="string",
        encoding="utf-8-sig",
        low_memory=False
    )


raw_data = {}

for table_name, file_path in FILE_PATHS.items():
    separator = REVIEWS_SEPARATOR if table_name == "reviews" else ","
    raw_data[table_name] = read_dataset(file_path, separator)

print("Dataset berhasil dibaca:")
for table_name, dataframe in raw_data.items():
    print(f"- {table_name}: {dataframe.shape[0]:,} baris x {dataframe.shape[1]} kolom")


Dataset berhasil dibaca:
- orders: 99,441 baris x 8 kolom
- order_items: 112,650 baris x 7 kolom
- products: 32,951 baris x 9 kolom
- category_translation: 71 baris x 2 kolom
- payments: 103,886 baris x 5 kolom
- reviews: 99,224 baris x 7 kolom


In [5]:
REQUIRED_COLUMNS = {
    "orders": [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "order_id", "order_item_id", "product_id",
        "price", "freight_value"
    ],
    "products": ["product_id", "product_category_name"],
    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],
    "payments": [
        "order_id", "payment_sequential",
        "payment_type", "payment_installments", "payment_value"
    ],
    "reviews": [
        "review_id", "order_id", "review_score",
        "review_creation_date", "review_answer_timestamp"
    ],
}

for table_name, required in REQUIRED_COLUMNS.items():
    missing_columns = [
        column for column in required
        if column not in raw_data[table_name].columns
    ]

    if missing_columns:
        raise ValueError(
            f"Kolom wajib tidak ditemukan pada {table_name}: {missing_columns}"
        )

print("Pemeriksaan kolom wajib selesai. Semua kolom yang dibutuhkan tersedia.")


Pemeriksaan kolom wajib selesai. Semua kolom yang dibutuhkan tersedia.


## 3. Pemeriksaan struktur dan nilai kosong

Pemeriksaan dilakukan sebelum konversi tipe data. Nilai kosong dilaporkan per kolom, bukan langsung diisi atau dihapus.

Nilai kosong pada kolom komentar review, misalnya, tidak otomatis berarti record review tidak valid. Penanganannya bergantung pada analisis yang akan dilakukan.


In [6]:
structure_report = pd.DataFrame([
    {
        "table": table_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "missing_cells": int(dataframe.isna().sum().sum()),
        "duplicate_full_rows": int(dataframe.duplicated().sum()),
    }
    for table_name, dataframe in raw_data.items()
])

display(structure_report)


,table,rows,columns,missing_cells,duplicate_full_rows
0,orders,99441,8,4908,0
1,order_items,112650,7,0,0
2,products,32951,9,2448,0
3,category_translation,71,2,0,0
4,payments,103886,5,0,0
5,reviews,99224,7,145903,0


In [10]:
missing_reports = []

for table_name, dataframe in raw_data.items():
    table_missing = dataframe.isna().sum()
    table_missing_pct = (dataframe.isna().mean() * 100).round(2)

    for column in dataframe.columns:
        missing_reports.append({
            "table": table_name,
            "column": column,
            "missing_values": int(table_missing[column]),
            "missing_pct": float(table_missing_pct[column]),
        })

missing_report = (
    pd.DataFrame(missing_reports)
    .sort_values(
        ["table", "missing_values"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(
    missing_report[missing_report["missing_values"] > 0]
)


,table,column,missing_values,missing_pct
9,orders,order_delivered_customer_date,2965,2.98
10,orders,order_delivered_carrier_date,1783,1.79
11,orders,order_approved_at,160,0.16
22,products,product_category_name,610,1.85
23,products,product_name_lenght,610,1.85
24,products,product_description_lenght,610,1.85
25,products,product_photos_qty,610,1.85
26,products,product_weight_g,2,0.01
27,products,product_length_cm,2,0.01
28,products,product_height_cm,2,0.01


## 4. Pemeriksaan duplikasi

Pemeriksaan mencakup duplikasi seluruh baris dan duplikasi berdasarkan kandidat kunci tabel. Hasil ini merupakan indikator untuk investigasi, bukan instruksi untuk langsung menghapus data.

Kandidat kunci yang diperiksa:
- `orders`: `order_id`
- `order_items`: kombinasi `order_id` dan `order_item_id`
- `products`: `product_id`
- `category_translation`: `product_category_name`
- `payments`: kombinasi `order_id` dan `payment_sequential`
- `reviews`: `review_id`


In [11]:
KEY_COLUMNS = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id"],
}

duplicate_reports = []

for table_name, dataframe in raw_data.items():
    key_columns = KEY_COLUMNS[table_name]

    duplicate_reports.append({
        "table": table_name,
        "exact_duplicate_rows": int(dataframe.duplicated().sum()),
        "rows_in_duplicate_keys": int(
            dataframe.duplicated(subset=key_columns, keep=False).sum()
        ),
        "rows_with_missing_key": int(
            dataframe[key_columns].isna().any(axis=1).sum()
        ),
        "candidate_key": " + ".join(key_columns),
    })

duplicate_report = pd.DataFrame(duplicate_reports)
display(duplicate_report)


,table,exact_duplicate_rows,rows_in_duplicate_keys,rows_with_missing_key,candidate_key
0,orders,0,0,0,order_id
1,order_items,0,0,0,order_id + order_item_id
2,products,0,0,0,product_id
3,category_translation,0,0,0,product_category_name
4,payments,0,0,0,order_id + payment_sequential
5,reviews,0,1603,0,review_id


In [13]:
# Investigasi duplikasi review_id

reviews = raw_data["reviews"]

duplicate_reviews = (
    reviews[
        reviews.duplicated(
            subset=["review_id"],
            keep=False
        )
    ]
    .sort_values(["review_id", "order_id"])
    .copy()
)

print("Jumlah baris dalam kelompok duplikasi:",
      len(duplicate_reviews))

print("Jumlah review_id yang berulang:",
      duplicate_reviews["review_id"].nunique())

# Ringkasan karakteristik setiap review_id berulang
review_duplicate_summary = (
    duplicate_reviews
    .groupby("review_id")
    .agg(
        total_rows=("review_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", "nunique"),
        unique_creation_dates=("review_creation_date", "nunique"),
        unique_answer_dates=("review_answer_timestamp", "nunique"),
        unique_titles=("review_comment_title", "nunique"),
        unique_messages=("review_comment_message", "nunique")
    )
    .reset_index()
    .sort_values("total_rows", ascending=False)
)

display(review_duplicate_summary.head(20))

# Menampilkan contoh baris yang mengalami duplikasi
display(duplicate_reviews.head(30))


Jumlah baris dalam kelompok duplikasi: 1603
Jumlah review_id yang berulang: 789


,review_id,total_rows,unique_orders,unique_scores,unique_creation_dates,unique_answer_dates,unique_titles,unique_messages
323,69a1068c3128a14994e3e422e4539e04,3,3,1,1,1,0,0
133,2d6ac45f859465b5c185274a1c929637,3,3,1,1,1,0,1
35,0c76e7a547a531e7bf9f0b99cba071c1,3,3,1,1,1,0,0
703,e44840754f12fad2b8646712121b349a,3,3,1,1,1,0,0
170,38821b5c496b678cf91acc34892805ad,3,3,1,1,1,0,0
137,308316408775d1600dad81bd3184556d,3,3,1,1,1,0,1
614,c444278834184f72b1484dfe47de7f97,3,3,1,1,1,0,0
761,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,3,3,1,1,1,0,0
229,4d0e6dd087008d1f992d25ef6e1f619f,3,3,1,1,1,0,0
142,32415bbf6e341d5d517080a796f79b5c,3,3,1,1,1,1,1


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,<NA>,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,<NA>,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,<NA>,<NA>,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,<NA>,<NA>,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,<NA>,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,<NA>,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,<NA>,<NA>,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,<NA>,<NA>,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,<NA>,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,<NA>,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [15]:

# Investigasi lanjutan duplikasi pada tabel reviews

reviews = raw_data["reviews"].copy()

# 1. Pemeriksaan konsistensi review_id
review_id_audit = (
    reviews.groupby("review_id", dropna=False)
    .agg(
        total_rows=("order_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", lambda x: x.nunique(dropna=False)),
        unique_creation_dates=("review_creation_date", lambda x: x.nunique(dropna=False)),
        unique_answer_dates=("review_answer_timestamp", lambda x: x.nunique(dropna=False)),
        unique_titles=("review_comment_title", lambda x: x.nunique(dropna=False)),
        unique_messages=("review_comment_message", lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

review_id_audit["content_conflict"] = (
    (review_id_audit["unique_scores"] > 1) |
    (review_id_audit["unique_creation_dates"] > 1) |
    (review_id_audit["unique_answer_dates"] > 1) |
    (review_id_audit["unique_titles"] > 1) |
    (review_id_audit["unique_messages"] > 1)
)

repeated_review_ids = review_id_audit[
    review_id_audit["total_rows"] > 1
]

print("=== AUDIT REVIEW_ID ===")
print("Jumlah review_id berulang:", len(repeated_review_ids))
print("Review ID pada beberapa order:",
      int((repeated_review_ids["unique_orders"] > 1).sum()))
print("Review ID dengan atribut berbeda:",
      int(repeated_review_ids["content_conflict"].sum()))

display(repeated_review_ids.head(10))


# 2. Pemeriksaan duplikasi berdasarkan order_id
order_id_audit = (
    reviews.groupby("order_id", dropna=False)
    .agg(
        total_rows=("review_id", "size"),
        unique_review_ids=("review_id", lambda x: x.nunique(dropna=False)),
        unique_scores=("review_score", lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

repeated_order_ids = order_id_audit[
    order_id_audit["total_rows"] > 1
]

print("\n=== AUDIT ORDER_ID ===")
print("Jumlah order_id berulang:", len(repeated_order_ids))
print("Jumlah baris dalam kelompok order berulang:",
      int(repeated_order_ids["total_rows"].sum()))
print("Order dengan beberapa review_id:",
      int((repeated_order_ids["unique_review_ids"] > 1).sum()))

display(repeated_order_ids.head(20))


# 3. Pemeriksaan duplikasi seluruh kolom
print("\n=== DUPLIKASI SELURUH BARIS ===")
print("Jumlah duplikasi seluruh baris:",
      int(reviews.duplicated().sum()))


=== AUDIT REVIEW_ID ===
Jumlah review_id berulang: 789
Review ID pada beberapa order: 789
Review ID dengan atribut berbeda: 0


,review_id,total_rows,unique_orders,unique_scores,unique_creation_dates,unique_answer_dates,unique_titles,unique_messages,content_conflict
31,00130cbe1f9d422698c812ed8ded1919,2,2,1,1,1,1,1,False
425,0115633a9c298b6a98bcbe4eee75345f,2,2,1,1,1,1,1,False
568,0174caf0ee5964646040cd94e15ac95e,2,2,1,1,1,1,1,False
574,017808d29fd1f942d97e50184dfb4c13,2,2,1,1,1,1,1,False
884,0254bd905dc677a6078990aad3331a36,2,2,1,1,1,1,1,False
961,0288d42bef3dfe36930740c9588a570f,2,2,1,1,1,1,1,False
1000,02aa7f5f75e964e3c7efa59a1f515281,2,2,1,1,1,1,1,False
1225,034528cb00b6bc981847acafbdf2ae0b,2,2,1,1,1,1,1,False
1364,03a6a25db577d0689440933055111897,2,2,1,1,1,1,1,False
1674,0467560f511c516ddaa54a60edb0c291,2,2,1,1,1,1,1,False



=== AUDIT ORDER_ID ===
Jumlah order_id berulang: 547
Jumlah baris dalam kelompok order berulang: 1098
Order dengan beberapa review_id: 547


,order_id,total_rows,unique_review_ids,unique_scores
84,0035246a40f520710769010f752e7507,2,2,1
461,013056cfe49763c6f66bda03396c5ee3,2,2,2
556,0176a6846bcb3b0d3aa3116a9a768597,2,2,1
835,02355020fd0a40a0d56df9f6ff060413,2,2,2
985,029863af4b968de1e5d6a82782e662f5,2,2,2
1092,02e0b68852217f5715fb9cc885829454,2,2,1
1103,02e723e8edb4a123d414f56cc9c4665e,2,2,1
1272,03515a836bb855b03f7df9dee520a8fc,2,2,1
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,3,2
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,2



=== DUPLIKASI SELURUH BARIS ===
Jumlah duplikasi seluruh baris: 0


In [16]:

# Investigasi lanjutan: validasi kombinasi kunci reviews

reviews = raw_data["reviews"].copy()

# 1. Pemeriksaan kombinasi order_id dan review_id
duplicate_pairs = reviews.duplicated(
    subset=["order_id", "review_id"],
    keep=False
)

print("=== PEMERIKSAAN KUNCI GABUNGAN ===")
print(
    "Jumlah baris dengan kombinasi order_id + review_id berulang:",
    int(duplicate_pairs.sum())
)

print(
    "Jumlah duplikasi tambahan:",
    int(reviews.duplicated(
        subset=["order_id", "review_id"]
    ).sum())
)

# 2. Pemeriksaan order dengan skor berbeda
order_audit = (
    reviews.groupby("order_id", dropna=False)
    .agg(
        total_rows=("review_id", "size"),
        unique_review_ids=("review_id", lambda x: x.nunique(dropna=False)),
        unique_scores=("review_score", lambda x: x.nunique(dropna=False)),
        unique_creation_dates=("review_creation_date", lambda x: x.nunique(dropna=False)),
        unique_answer_dates=("review_answer_timestamp", lambda x: x.nunique(dropna=False)),
        unique_titles=("review_comment_title", lambda x: x.nunique(dropna=False)),
        unique_messages=("review_comment_message", lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

repeated_orders = order_audit[
    order_audit["total_rows"] > 1
]

conflicting_orders = repeated_orders[
    repeated_orders["unique_scores"] > 1
]

print("\n=== DISTRIBUSI JUMLAH REVIEW PER PESANAN ===")
print(repeated_orders["total_rows"].value_counts().sort_index())

print("\nJumlah pesanan dengan skor berbeda:",
      len(conflicting_orders))

# 3. Menampilkan contoh pesanan dengan skor berbeda
sample_order_ids = conflicting_orders["order_id"].head(5)

print("\n=== CONTOH REVIEW DENGAN SKOR BERBEDA ===")

display(
    reviews[
        reviews["order_id"].isin(sample_order_ids)
    ][
        [
            "order_id",
            "review_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].sort_values(["order_id", "review_creation_date"])
)

# 4. Menampilkan contoh review_id yang terhubung ke beberapa pesanan
sample_review_ids = (
    reviews.groupby("review_id")
    .filter(lambda x: len(x) > 1)["review_id"]
    .drop_duplicates()
    .head(5)
)

print("\n=== CONTOH REVIEW_ID PADA BEBERAPA PESANAN ===")

display(
    reviews[
        reviews["review_id"].isin(sample_review_ids)
    ][
        [
            "review_id",
            "order_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].sort_values("review_id")
)


=== PEMERIKSAAN KUNCI GABUNGAN ===
Jumlah baris dengan kombinasi order_id + review_id berulang: 0
Jumlah duplikasi tambahan: 0

=== DISTRIBUSI JUMLAH REVIEW PER PESANAN ===
total_rows
2    543
3      4
Name: count, dtype: int64

Jumlah pesanan dengan skor berbeda: 202

=== CONTOH REVIEW DENGAN SKOR BERBEDA ===


,order_id,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,<NA>,<NA>,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,<NA>,<NA>,2018-03-04 00:00:00,2018-03-05 17:02:00
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,<NA>,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,<NA>,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,<NA>,<NA>,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,<NA>,<NA>,2017-07-19 00:00:00,2017-07-20 12:06:11
69438,03c939fd7fd3b38f8485a0f95798f1f6,405eb2ea45e1dbe2662541ae5b47e2aa,3,<NA>,Seria ótimo se tivesem entregue os 3 (três) pe...,2018-03-06 00:00:00,2018-03-06 19:50:32
8273,03c939fd7fd3b38f8485a0f95798f1f6,b04ed893318da5b863e878cd3d0511df,3,<NA>,Um ponto negativo que achei foi a cobrança de ...,2018-03-20 00:00:00,2018-03-21 02:28:23
51527,03c939fd7fd3b38f8485a0f95798f1f6,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,4,<NA>,<NA>,2018-03-29 00:00:00,2018-03-30 00:29:09
88624,03eba6d9fef8f5b3e811d4b5a7cca9cd,36ce47fb903bec726f89c65eac26dc9f,4,<NA>,<NA>,2018-02-23 00:00:00,2018-02-23 23:55:53



=== CONTOH REVIEW_ID PADA BEBERAPA PESANAN ===


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
200,28642ce6250b94cc72bc85960aec6c62,e239d280236cdd3c40cb2c033f681d1c,5,2018-03-25 00:00:00,2018-03-25 21:03:02
58385,28642ce6250b94cc72bc85960aec6c62,bc42a955f289870d5789e6e437206300,5,2018-03-25 00:00:00,2018-03-25 21:03:02
393,6b1de94de0f4bd84dfc4136818242faa,92acf87839903a94aeca0e5040d99acb,5,2018-02-16 00:00:00,2018-02-19 19:04:21
12473,6b1de94de0f4bd84dfc4136818242faa,9de125e032c8c74ccbcf9a5b7a3b4faa,5,2018-02-16 00:00:00,2018-02-19 19:04:21
344,a0a641414ff718ca079b3967ef5c2495,169d7e0fd71d624d306f132acd791cbe,5,2018-03-04 00:00:00,2018-03-06 20:12:53
27356,a0a641414ff718ca079b3967ef5c2495,4e93b736e8d687bca088c6ee496437e8,5,2018-03-04 00:00:00,2018-03-06 20:12:53
360,ecbaf1fce7d2c09bfab46f89065afeaf,2451b9756f310d4cff5c7987b393870d,5,2017-07-27 00:00:00,2017-07-28 16:57:18
84556,ecbaf1fce7d2c09bfab46f89065afeaf,a67f04f0a3022835f86664a0070fe36e,5,2017-07-27 00:00:00,2017-07-28 16:57:18
346,f4d74b17cd63ee35efa82cd2567de911,f269e83a82f64baa3de97c2ebf3358f6,3,2018-01-12 00:00:00,2018-01-13 18:46:10
90298,f4d74b17cd63ee35efa82cd2567de911,a60c9bf6dcdd39e9f1c10ae5ad77c24d,3,2018-01-12 00:00:00,2018-01-13 18:46:10


### Kesimpulan investigasi duplikasi tabel reviews

Hasil pemeriksaan menunjukkan bahwa tabel reviews memiliki karakteristik hubungan data yang tidak sepenuhnya satu baris untuk satu pesanan.

Ditemukan 789 kelompok review_id yang muncul pada beberapa order_id berbeda. Seluruh atribut pada kelompok review_id yang berulang menunjukkan nilai konsisten.

Selain itu, terdapat 547 pesanan dengan lebih dari satu review, terdiri dari 543 pesanan dengan dua review dan 4 pesanan dengan tiga review. Sebanyak 202 pesanan memiliki skor review yang berbeda.

Pemeriksaan lebih lanjut menunjukkan bahwa tidak ditemukan duplikasi pada kombinasi order_id dan review_id maupun duplikasi seluruh baris.

**Keputusan pembersihan:**
- Seluruh baris pada tabel reviews dipertahankan.
- Tidak dilakukan penghapusan otomatis berdasarkan review_id maupun order_id.
- Kombinasi order_id dan review_id digunakan sebagai identitas unik catatan review pada dataset ini.
- Untuk analisis tingkat pesanan, data review perlu diagregasi terlebih dahulu agar tidak menyebabkan penggandaan baris ketika digabungkan dengan tabel orders.

**Catatan metodologis:** beberapa review dalam satu pesanan tidak otomatis dianggap sebagai kesalahan data. Perbedaan skor dan tanggal tetap dipertahankan agar informasi ulasan pelanggan tidak hilang.


## 5. Konversi tipe tanggal dan numerik

Kolom tanggal diubah ke tipe `datetime`. Kolom angka diubah ke tipe numerik. Nilai yang tidak dapat dikonversi menjadi `NaT` atau `NaN` dan jumlahnya dicatat.

Konversi tipe data tidak menghapus record.


In [17]:
cleaned_data = {
    table_name: dataframe.copy()
    for table_name, dataframe in raw_data.items()
}

DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

date_conversion_reports = []

for table_name, columns in DATE_COLUMNS.items():
    dataframe = cleaned_data[table_name]

    for column in columns:
        if column not in dataframe.columns:
            continue

        original = dataframe[column]
        converted = pd.to_datetime(original, errors="coerce")

        invalid_mask = original.notna() & converted.isna()

        date_conversion_reports.append({
            "table": table_name,
            "column": column,
            "non_empty_before": int(original.notna().sum()),
            "invalid_date_values": int(invalid_mask.sum()),
        })

        dataframe[column] = converted

date_conversion_report = pd.DataFrame(date_conversion_reports)
display(date_conversion_report)


,table,column,non_empty_before,invalid_date_values
0,orders,order_purchase_timestamp,99441,0
1,orders,order_approved_at,99281,0
2,orders,order_delivered_carrier_date,97658,0
3,orders,order_delivered_customer_date,96476,0
4,orders,order_estimated_delivery_date,99441,0
5,order_items,shipping_limit_date,112650,0
6,reviews,review_creation_date,99224,0
7,reviews,review_answer_timestamp,99224,0


In [18]:
NUMERIC_COLUMNS = {
    "order_items": [
        "order_item_id", "price", "freight_value"
    ],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value",
    ],
    "reviews": ["review_score"],
}

numeric_conversion_reports = []

for table_name, columns in NUMERIC_COLUMNS.items():
    dataframe = cleaned_data[table_name]

    for column in columns:
        if column not in dataframe.columns:
            continue

        original = dataframe[column]
        converted = pd.to_numeric(original, errors="coerce")

        invalid_mask = original.notna() & converted.isna()

        numeric_conversion_reports.append({
            "table": table_name,
            "column": column,
            "non_empty_before": int(original.notna().sum()),
            "invalid_numeric_values": int(invalid_mask.sum()),
        })

        dataframe[column] = converted

numeric_conversion_report = pd.DataFrame(numeric_conversion_reports)
display(numeric_conversion_report)


,table,column,non_empty_before,invalid_numeric_values
0,order_items,order_item_id,112650,0
1,order_items,price,112650,0
2,order_items,freight_value,112650,0
3,products,product_name_lenght,32341,0
4,products,product_description_lenght,32341,0
5,products,product_photos_qty,32341,0
6,products,product_weight_g,32949,0
7,products,product_length_cm,32949,0
8,products,product_height_cm,32949,0
9,products,product_width_cm,32949,0


## 6. Membuat kolom turunan untuk analisis

Kolom turunan dibuat pada tabel pesanan:
- `purchase_month`: bulan pembelian dalam format `YYYY-MM`.
- `delivery_duration_days`: selisih hari kalender antara pembelian dan penerimaan barang.
- `delivery_delay_days`: selisih hari kalender antara penerimaan aktual dan estimasi.
- `is_late`: `1` jika terlambat, `0` jika tepat waktu/lebih awal, dan kosong jika tidak memenuhi kriteria analisis.
- `delivery_group`: label keterlambatan.

Untuk analisis keterlambatan, pesanan harus berstatus `delivered`, memiliki tanggal pembelian, tanggal diterima, dan estimasi pengiriman, serta tidak memiliki urutan tanggal pengiriman yang tidak logis. Record yang tidak memenuhi syarat tetap disimpan; hanya tidak diberi label keterlambatan.


In [27]:

# ============================================================
# 6. MEMBUAT KOLOM TURUNAN UNTUK ANALISIS
# ============================================================

orders_clean = cleaned_data["orders"]

# ------------------------------------------------------------
# 1. Mengambil kolom tanggal
# ------------------------------------------------------------

purchase = orders_clean["order_purchase_timestamp"]
carrier = orders_clean["order_delivered_carrier_date"]
delivered = orders_clean["order_delivered_customer_date"]
estimated = orders_clean["order_estimated_delivery_date"]


# ------------------------------------------------------------
# 2. Membuat kolom bulan pembelian
# ------------------------------------------------------------

orders_clean["purchase_month"] = (
    purchase.dt.to_period("M").astype("string")
)


# ------------------------------------------------------------
# 3. Menghitung lama pengiriman
#    Tanggal diterima - tanggal pembelian
# ------------------------------------------------------------

valid_duration = (
    orders_clean["order_status"].eq("delivered")
    & purchase.notna()
    & delivered.notna()
    & delivered.ge(purchase)
)

delivery_duration = (
    delivered.dt.normalize() - purchase.dt.normalize()
).dt.days

orders_clean["delivery_duration_days"] = (
    delivery_duration.where(valid_duration).astype("Int64")
)


# ------------------------------------------------------------
# 4. Memeriksa urutan tanggal yang tidak valid
# ------------------------------------------------------------

carrier_before_purchase = (
    carrier.notna()
    & purchase.notna()
    & carrier.lt(purchase)
)

customer_before_carrier = (
    delivered.notna()
    & carrier.notna()
    & delivered.lt(carrier)
)

delivered_before_purchase = (
    delivered.notna()
    & purchase.notna()
    & delivered.lt(purchase)
)

estimated_before_purchase = (
    estimated.notna()
    & purchase.notna()
    & estimated.lt(purchase)
)


# ------------------------------------------------------------
# 5. Menentukan pesanan yang memenuhi kriteria analisis
# ------------------------------------------------------------

eligible_delivery = (
    orders_clean["order_status"].eq("delivered")
    & purchase.notna()
    & delivered.notna()
    & estimated.notna()
    & carrier.notna()
    & ~carrier_before_purchase
    & ~customer_before_carrier
    & ~delivered_before_purchase
)


# ------------------------------------------------------------
# 6. Menghitung keterlambatan pengiriman
#    Tanggal diterima - tanggal estimasi
# ------------------------------------------------------------

delivery_delay = (
    delivered.dt.normalize() - estimated.dt.normalize()
).dt.days

orders_clean["delivery_delay_days"] = (
    delivery_delay.where(eligible_delivery).astype("Int64")
)


# ------------------------------------------------------------
# 7. Membuat label keterlambatan
#    1 = Terlambat
#    0 = Tepat waktu atau lebih cepat
#    NA = Tidak memenuhi kriteria analisis
# ------------------------------------------------------------

orders_clean["is_late"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="Int64"
)

orders_clean.loc[eligible_delivery, "is_late"] = (
    delivery_delay.loc[eligible_delivery]
    .gt(0)
    .astype("Int64")
)


# ------------------------------------------------------------
# 8. Membuat kelompok pengiriman
# ------------------------------------------------------------

orders_clean["delivery_group"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="string"
)

late_mask = (
    eligible_delivery
    & orders_clean["is_late"].eq(1).fillna(False)
)

ontime_mask = (
    eligible_delivery
    & orders_clean["is_late"].eq(0).fillna(False)
)

orders_clean.loc[late_mask, "delivery_group"] = "Late"

orders_clean.loc[ontime_mask, "delivery_group"] = (
    "On time / early"
)


# ------------------------------------------------------------
# 9. Menampilkan ringkasan hasil
# ------------------------------------------------------------

print(f"Jumlah seluruh pesanan: {len(orders_clean):,}")

print(
    "Pesanan delivered yang memenuhi kriteria analisis "
    f"keterlambatan: {int(eligible_delivery.sum()):,}"
)

print("\nDistribusi kelompok pengiriman:")

display(
    orders_clean["delivery_group"]
    .value_counts(dropna=False)
    .rename_axis("delivery_group")
    .reset_index(name="orders")
)

print("\nPembuatan kolom turunan selesai.")


Jumlah seluruh pesanan: 99,441
Pesanan delivered yang memenuhi kriteria analisis keterlambatan: 96,281

Distribusi kelompok pengiriman:


,delivery_group,orders
0,On time / early,89750
1,Late,6531
2,<NA>,3160



Pembuatan kolom turunan selesai.


In [28]:

# ============================================================
# AUDIT PESANAN YANG TIDAK TERKLASIFIKASI
# ============================================================

unclassified_mask = orders_clean["delivery_group"].isna()

audit_conditions = pd.DataFrame({
    "bukan_delivered": (
        ~orders_clean["order_status"].eq("delivered").fillna(False)
    ),
    "tanggal_pembelian_kosong": purchase.isna(),
    "tanggal_diterima_kosong": delivered.isna(),
    "tanggal_estimasi_kosong": estimated.isna(),
    "carrier_sebelum_pembelian": carrier_before_purchase,
    "diterima_sebelum_carrier": customer_before_carrier,
    "diterima_sebelum_pembelian": delivered_before_purchase,
    "estimasi_sebelum_pembelian": estimated_before_purchase
}, index=orders_clean.index)

# Ringkasan penyebab pesanan tidak terklasifikasi
audit_summary = (
    audit_conditions.loc[unclassified_mask]
    .sum()
    .astype(int)
    .rename("jumlah_pesanan")
    .to_frame()
    .sort_values("jumlah_pesanan", ascending=False)
)

print("=== AUDIT PESANAN TIDAK TERKLASIFIKASI ===")

print(
    f"Total pesanan tidak terklasifikasi: "
    f"{int(unclassified_mask.sum()):,}"
)

print(
    "Pesanan delivered yang tidak terklasifikasi:",
    int((
        unclassified_mask
        & orders_clean["order_status"].eq("delivered").fillna(False)
    ).sum())
)

display(audit_summary)

print(
    "\nCatatan: Satu pesanan dapat memiliki lebih dari "
    "satu kondisi anomali."
)


=== AUDIT PESANAN TIDAK TERKLASIFIKASI ===
Total pesanan tidak terklasifikasi: 3,160
Pesanan delivered yang tidak terklasifikasi: 197


,jumlah_pesanan
tanggal_diterima_kosong,2965
bukan_delivered,2963
carrier_sebelum_pembelian,166
diterima_sebelum_carrier,23
tanggal_estimasi_kosong,0
tanggal_pembelian_kosong,0
diterima_sebelum_pembelian,0
estimasi_sebelum_pembelian,0



Catatan: Satu pesanan dapat memiliki lebih dari satu kondisi anomali.


In [29]:

# ============================================================
# 6. MEMBUAT KOLOM TURUNAN UNTUK ANALISIS
# ============================================================

orders_clean = cleaned_data["orders"]

# ------------------------------------------------------------
# 1. Mengambil kolom tanggal
# ------------------------------------------------------------

purchase = orders_clean["order_purchase_timestamp"]
carrier = orders_clean["order_delivered_carrier_date"]
delivered = orders_clean["order_delivered_customer_date"]
estimated = orders_clean["order_estimated_delivery_date"]


# ------------------------------------------------------------
# 2. Membuat kolom bulan pembelian
# ------------------------------------------------------------

orders_clean["purchase_month"] = (
    purchase.dt.to_period("M").astype("string")
)


# ------------------------------------------------------------
# 3. Pemeriksaan urutan tanggal
# ------------------------------------------------------------

carrier_before_purchase = (
    carrier.notna()
    & purchase.notna()
    & carrier.lt(purchase)
)

customer_before_carrier = (
    delivered.notna()
    & carrier.notna()
    & delivered.lt(carrier)
)

delivered_before_purchase = (
    delivered.notna()
    & purchase.notna()
    & delivered.lt(purchase)
)

estimated_before_purchase = (
    estimated.notna()
    & purchase.notna()
    & estimated.lt(purchase)
)

# Simpan indikator anomali untuk dokumentasi
orders_clean["carrier_before_purchase"] = (
    carrier_before_purchase.astype("boolean")
)

orders_clean["delivered_before_carrier"] = (
    customer_before_carrier.astype("boolean")
)


# ------------------------------------------------------------
# 4. Menghitung lama pengiriman
#    Tanggal diterima - tanggal pembelian
# ------------------------------------------------------------

valid_duration = (
    orders_clean["order_status"].eq("delivered")
    & purchase.notna()
    & delivered.notna()
    & delivered.ge(purchase)
    & ~customer_before_carrier
)

delivery_duration = (
    delivered.dt.normalize()
    - purchase.dt.normalize()
).dt.days

orders_clean["delivery_duration_days"] = (
    delivery_duration
    .where(valid_duration)
    .astype("Int64")
)


# ------------------------------------------------------------
# 5. Menentukan pesanan yang layak untuk analisis keterlambatan
# ------------------------------------------------------------

eligible_delivery = (
    orders_clean["order_status"].eq("delivered")
    & purchase.notna()
    & delivered.notna()
    & estimated.notna()
    & delivered.ge(purchase)
    & estimated.ge(purchase)
    & ~customer_before_carrier
)

# Catatan:
# carrier_before_purchase tidak digunakan sebagai
# kriteria pengecualian karena metrik keterlambatan
# menggunakan tanggal penerimaan dan estimasi.


# ------------------------------------------------------------
# 6. Menghitung keterlambatan pengiriman
#    Tanggal diterima - tanggal estimasi
# ------------------------------------------------------------

delivery_delay = (
    delivered.dt.normalize()
    - estimated.dt.normalize()
).dt.days

orders_clean["delivery_delay_days"] = (
    delivery_delay
    .where(eligible_delivery)
    .astype("Int64")
)


# ------------------------------------------------------------
# 7. Membuat label keterlambatan
#    1 = Terlambat
#    0 = Tepat waktu atau lebih cepat
#    NA = Tidak memenuhi kriteria analisis
# ------------------------------------------------------------

orders_clean["is_late"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="Int64"
)

orders_clean.loc[eligible_delivery, "is_late"] = (
    delivery_delay.loc[eligible_delivery]
    .gt(0)
    .astype("Int64")
)


# ------------------------------------------------------------
# 8. Membuat kelompok pengiriman
# ------------------------------------------------------------

orders_clean["delivery_group"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="string"
)

late_mask = (
    eligible_delivery
    & orders_clean["is_late"].eq(1).fillna(False)
)

ontime_mask = (
    eligible_delivery
    & orders_clean["is_late"].eq(0).fillna(False)
)

orders_clean.loc[late_mask, "delivery_group"] = "Late"

orders_clean.loc[ontime_mask, "delivery_group"] = (
    "On time / early"
)


# ------------------------------------------------------------
# 9. Ringkasan hasil analisis
# ------------------------------------------------------------

total_orders = len(orders_clean)
eligible_count = int(eligible_delivery.sum())

late_count = int(late_mask.sum())
ontime_count = int(ontime_mask.sum())
unclassified_count = int(
    orders_clean["delivery_group"].isna().sum()
)

print("=== RINGKASAN ANALISIS PENGIRIMAN ===")

print(f"Jumlah seluruh pesanan: {total_orders:,}")

print(
    "Pesanan memenuhi kriteria analisis: "
    f"{eligible_count:,}"
)

print(
    "Pesanan Late: "
    f"{late_count:,}"
)

print(
    "Pesanan On time / early: "
    f"{ontime_count:,}"
)

print(
    "Pesanan tidak terklasifikasi: "
    f"{unclassified_count:,}"
)

if eligible_count > 0:
    print(
        "Persentase keterlambatan: "
        f"{late_count / eligible_count * 100:.2f}%"
    )

print("\nDistribusi kelompok pengiriman:")

display(
    orders_clean["delivery_group"]
    .value_counts(dropna=False)
    .rename_axis("delivery_group")
    .reset_index(name="orders")
)

print("\nPembuatan kolom turunan selesai.")


=== RINGKASAN ANALISIS PENGIRIMAN ===
Jumlah seluruh pesanan: 99,441
Pesanan memenuhi kriteria analisis: 96,447
Pesanan Late: 6,534
Pesanan On time / early: 89,913
Pesanan tidak terklasifikasi: 2,994
Persentase keterlambatan: 6.77%

Distribusi kelompok pengiriman:


,delivery_group,orders
0,On time / early,89913
1,Late,6534
2,<NA>,2994



Pembuatan kolom turunan selesai.


## 7. Validasi hasil pembersihan

Validasi memastikan proses konversi dan pembuatan kolom turunan tidak mengurangi jumlah baris sumber. Duplikasi dan nilai kosong tetap tersedia untuk ditinjau sesuai kebutuhan analisis.


In [30]:
row_count_validation = pd.DataFrame([
    {
        "table": table_name,
        "rows_before": len(raw_data[table_name]),
        "rows_after": len(cleaned_data[table_name]),
        "row_count_preserved": (
            len(raw_data[table_name]) == len(cleaned_data[table_name])
        ),
    }
    for table_name in raw_data
])

display(row_count_validation)

assert row_count_validation["row_count_preserved"].all(), (
    "Jumlah baris berubah. Periksa proses pembersihan sebelum melanjutkan."
)

print("Validasi jumlah baris berhasil: tidak ada record yang dihapus.")


,table,rows_before,rows_after,row_count_preserved
0,orders,99441,99441,True
1,order_items,112650,112650,True
2,products,32951,32951,True
3,category_translation,71,71,True
4,payments,103886,103886,True
5,reviews,99224,99224,True


Validasi jumlah baris berhasil: tidak ada record yang dihapus.


## 8. Menyimpan dataset hasil pembersihan

Dataset hasil pembersihan disimpan di `Dataset/Processed/Cleaned`. File sumber di `Dataset/Raw` tidak ditimpa.

File output akan digunakan kembali pada notebook `02_eksplorasi.ipynb`, sehingga proses konversi dan pembersihan tidak perlu diulang di notebook EDA.


In [31]:
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_NAMES = {
    "orders": "orders_python_cleaned.csv",
    "order_items": "order_items_python_cleaned.csv",
    "products": "products_python_cleaned.csv",
    "category_translation": "category_translation_python_cleaned.csv",
    "payments": "payments_python_cleaned.csv",
    "reviews": "reviews_python_cleaned.csv",
}

for table_name, output_name in OUTPUT_NAMES.items():
    output_path = CLEANED_DIR / output_name

    cleaned_data[table_name].to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved: {output_path.name}")

# Simpan laporan pemeriksaan agar keputusan pembersihan terdokumentasi.
missing_report.to_csv(
    CLEANED_DIR / "missing_values_report.csv",
    index=False,
    encoding="utf-8-sig"
)

duplicate_report.to_csv(
    CLEANED_DIR / "duplicate_check_report.csv",
    index=False,
    encoding="utf-8-sig"
)

date_conversion_report.to_csv(
    CLEANED_DIR / "date_conversion_report.csv",
    index=False,
    encoding="utf-8-sig"
)

numeric_conversion_report.to_csv(
    CLEANED_DIR / "numeric_conversion_report.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSemua dataset dan laporan berhasil disimpan.")


Saved: orders_python_cleaned.csv
Saved: order_items_python_cleaned.csv
Saved: products_python_cleaned.csv
Saved: category_translation_python_cleaned.csv
Saved: payments_python_cleaned.csv
Saved: reviews_python_cleaned.csv

Semua dataset dan laporan berhasil disimpan.


## 9. Ringkasan keputusan pembersihan

| Area | Keputusan |
|---|---|
| File sumber | Tidak diubah; seluruh hasil disimpan sebagai salinan di folder `Processed/Cleaned`. |
| Nilai kosong | Dilaporkan. Tidak dilakukan imputasi massal atau penghapusan otomatis. |
| Duplikasi | Dilaporkan berdasarkan baris penuh dan kandidat kunci. Tidak dihapus otomatis. |
| Tipe tanggal | Dikonversi ke `datetime`; nilai yang gagal dikonversi menjadi `NaT` dan dicatat. |
| Tipe numerik | Dikonversi ke numerik; nilai yang gagal dikonversi menjadi `NaN` dan dicatat. |
| Bulan pembelian | Dibuat dari `order_purchase_timestamp` dengan format `YYYY-MM`. |
| Lama pengiriman | Dihitung sebagai selisih hari kalender antara pembelian dan penerimaan untuk pesanan delivered dengan urutan tanggal valid. |
| Keterlambatan | Hanya diberi label jika pesanan delivered memiliki tanggal aktual dan estimasi yang valid serta urutan tanggal logis. |
| Pendapatan | Pada tahap analisis, gunakan order berstatus `delivered`; nilai merchandise dihitung dari `price`, sedangkan freight dipisahkan. |

**Catatan metodologis:** nilai `is_late` yang kosong berarti pesanan tidak memenuhi kriteria analisis keterlambatan, bukan berarti pesanan tersebut tepat waktu. Pada tahap Minggu 4, hasil Python perlu dibandingkan dengan query SQL dan perbedaan definisi atau filter harus dijelaskan.


In [32]:
# Ringkasan akhir untuk dokumentasi
final_summary = pd.DataFrame([
    {
        "table": table_name,
        "rows_saved": len(cleaned_data[table_name]),
        "columns_saved": len(cleaned_data[table_name].columns),
        "output_file": OUTPUT_NAMES[table_name],
    }
    for table_name in cleaned_data
])

display(final_summary)

print("Notebook 01 — Pembersihan Data selesai.")
print("Lokasi output:", CLEANED_DIR)


,table,rows_saved,columns_saved,output_file
0,orders,99441,15,orders_python_cleaned.csv
1,order_items,112650,7,order_items_python_cleaned.csv
2,products,32951,9,products_python_cleaned.csv
3,category_translation,71,2,category_translation_python_cleaned.csv
4,payments,103886,5,payments_python_cleaned.csv
5,reviews,99224,7,reviews_python_cleaned.csv


Notebook 01 — Pembersihan Data selesai.
Lokasi output: /mnt/c/Users/ASUS/Github/Mini Project Olist/Dataset/Processed/Cleaned
